In [1]:
# ============================================================
# FILE 8 — Batch Inference & Earth Engine Asset Export
# Automatically processes all converted CSVs and submits tasks
# ============================================================

import os
import re
import time
import glob
import tempfile
import joblib
import ee
from geemap import ml



import yaml

# ------------------------------------------------------------
# 1. Load Configuration
# ------------------------------------------------------------
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

PROJECT_ID = config['gee']['project_id']
START_DATE = config['parameters']['start_date']
END_DATE = config['parameters']['end_date']
INPUT_FOLDER = config['paths']['models_joblib_dir']
OUTPUT_FOLDER = config['paths']['models_csv_dir']
CROP_LABELS = config['parameters']['crop_labels']



# ------------------------------------------------------------
# 1. Initialize Earth Engine
# ------------------------------------------------------------
try:
    ee.Initialize(project=PROJECT_ID, opt_url='https://earthengine-highvolume.googleapis.com')
    print('✅ Google Earth Engine initialized successfully!\n')
except ee.EEException as e:
    print('❌ Google Earth Engine failed to initialize!', e)
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID, opt_url='https://earthengine-highvolume.googleapis.com')


# Expected filename format: rfr_model_YYYY-MM-DD_AEZ_<N>_<VAR>.joblib
FILE_PATTERN = re.compile(r"rfr_model_(.+)_AEZ_(\d+)_(.+)\.joblib$")

# ------------------------------------------------------------
# 3. Reusable Helper Functions (defined once, outside the loop)
# ------------------------------------------------------------
def csv_to_classifier_safe(csv_path):
    """Converts CSV to GEE classifier, fixing hyphenated band names in tree logic."""
    with open(csv_path, 'r', encoding='utf-8') as fh:
        csv_text = fh.read()

    csv_text_fixed = csv_text.replace('pH_0-5', 'pH_0_5').replace('pH_5-15', 'pH_5_15')

    fd, tmp_path = tempfile.mkstemp(suffix='_fixed.csv', dir=os.path.dirname(csv_path))
    os.close(fd)
    with open(tmp_path, 'w', encoding='utf-8') as fh:
        fh.write(csv_text_fixed)
    try:
        classifier = ml.csv_to_classifier(tmp_path)
    finally:
        try:
            os.remove(tmp_path)
        except Exception:
            pass
    return classifier


def scale_landsat(image):
    qa = image.select('QA_PIXEL')
    mask = (qa.bitwiseAnd(1 << 3).eq(0)
              .And(qa.bitwiseAnd(1 << 4).eq(0))
              .And(qa.bitwiseAnd(1 << 2).eq(0))
              .And(qa.bitwiseAnd(1 << 5).eq(0)))
    scaled = (image.select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'])
                   .multiply(0.0000275).add(-0.2)
                   .rename(['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']))
    return image.addBands(scaled).updateMask(mask).copyProperties(image, ['system:time_start'])


def add_seasonal_indices(image):
    ndvi = image.normalizedDifference(['NIR', 'RED']).rename('NDVI')
    nirv = ndvi.multiply(image.select('NIR')).rename('NIRv')
    return image.addBands([ndvi, nirv])


def add_derived_indices(image):
    si   = image.normalizedDifference(['RED', 'BLUE']).rename('SI')
    ri   = image.expression('RED**2 / (BLUE * GREEN**3)', {
               'RED': image.select('RED'),
               'BLUE': image.select('BLUE'),
               'GREEN': image.select('GREEN')
           }).rename('RI')
    tgsi = image.normalizedDifference(['SWIR1', 'NIR']).rename('TGSI')
    return image.addBands([si, ri, tgsi])


def load_soil_image(property_name, features, roi):
    band_maps = {
        'sand':  {'sand05':  'sand_0-5cm_mean',  'sand515':  'sand_5-15cm_mean'},
        'silt':  {'silt05':  'silt_0-5cm_mean',  'silt515':  'silt_5-15cm_mean'},
        'clay':  {'clay05':  'clay_0-5cm_mean',  'clay515':  'clay_5-15cm_mean'},
        'phh2o': {'pH_0-5':  'phh2o_0-5cm_mean', 'pH_5-15':  'phh2o_5-15cm_mean'}
    }
    band_map = band_maps.get(property_name, {})
    selected_bands, new_names = [], []
    for model_feat, ee_band in band_map.items():
        if model_feat in features:
            selected_bands.append(ee_band)
            new_names.append(model_feat.replace('-', '_'))

    if selected_bands:
        img = (ee.Image(f"projects/soilgrids-isric/{property_name}_mean")
                 .select(selected_bands)
                 .rename(new_names)
                 .clip(roi))
        if property_name == 'phh2o':
            img = img.divide(10.0)
        return img
    return None



# ------------------------------------------------------------
# 4. Load AEZ Feature Collection Once (outside the loop)
# ------------------------------------------------------------
aez_fc = ee.FeatureCollection(config['gee']['aez_asset_path'])

# ------------------------------------------------------------
# 5. Batch Processing Engine
# ------------------------------------------------------------
joblib_files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.joblib")))
print(f"Found {len(joblib_files)} model file(s) for inference.\n")

if not joblib_files:
    raise FileNotFoundError(
        f"No .joblib files found in {INPUT_FOLDER}. Check your INPUT_FOLDER path."
    )

submitted_count = 0
skipped_count   = 0
failed_count    = 0

for file_path in joblib_files:
    filename = os.path.basename(file_path)

    # --- Parse Metadata from Filename ---
    match = FILE_PATTERN.match(filename)
    if not match:
        print(f"⚠️  Skipping '{filename}': does not match expected naming convention.")
        failed_count += 1
        continue

    model_date, aez_str, pred_var = match.groups()
    AEZ = int(aez_str)

    print(f"{'='*60}")
    print(f"  AEZ: {AEZ}  |  Nutrient: {pred_var}  |  Date: {model_date}")
    print(f"{'='*60}")

    # --- Check if GEE Asset Already Exists (crash-resilient) ---
    ASSET_NAME = f"AEZ_{AEZ}_{pred_var}_{model_date.replace('-', '')}"
    ASSET_ID   = f"projects/{PROJECT_ID}/assets/{ASSET_NAME}"

    try:
        ee.data.getAsset(ASSET_ID)
        print(f"  ⏭️  Asset already exists in GEE — skipping: {ASSET_NAME}\n")
        skipped_count += 1
        continue
    except ee.EEException:
        pass  # Asset does not exist, proceed

    # --- Load Local Model ---
    try:
        rf = joblib.load(file_path)
    except Exception as e:
        print(f"  ❌ Failed to load model '{filename}': {e}\n")
        failed_count += 1
        continue

    features     = list(rf.feature_names_in_)
    n_estimators = rf.get_params()['n_estimators']
    max_depth    = rf.get_params()['max_depth']
    print(f"  Features ({len(features)}): {features}")

    # --- Locate Corresponding CSV ---
    csv_filename = f"rf_trees_t{n_estimators}_d{max_depth}_{model_date}_AEZ_{AEZ}_{pred_var}.csv"
    out_csv      = os.path.join(OUTPUT_FOLDER, csv_filename)

    if not os.path.exists(out_csv):
        print(f"  ❌ Required CSV not found: {csv_filename}. Run File 7 first. Skipping.\n")
        failed_count += 1
        continue

    # --- Build GEE Classifier from CSV ---
    try:
        classifier = csv_to_classifier_safe(out_csv)
        print(f"  ✅ Classifier loaded from: {csv_filename}")
    except Exception as e:
        print(f"  ❌ Failed to build classifier from CSV: {e}\n")
        failed_count += 1
        continue

    # --- Define Region of Interest ---
    roi = aez_fc.filter(ee.Filter.eq('ae_regcode', AEZ)).geometry()

    # --- Build Feature Image Stack ---
    print(f"  ⏳ Building feature stack...")

    LS = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
            .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
            .filterBounds(roi)
            .filterDate(START_DATE, END_DATE)
            .map(scale_landsat))

    images_to_stack = []

    # Seasonal NDVI / NIRv
    LS_seasonal = LS.map(add_seasonal_indices)
    seasons = {
        'Kharif': ('2023-07-01', '2023-10-31'),
        'Rabi':   ('2023-11-01', '2024-03-31'),
        'Zaid':   ('2024-04-01', '2024-06-30')
    }
    for band in ['NIRv', 'NDVI']:
        for season_name, (s_date, e_date) in seasons.items():
            feat_name = f"{band}_{season_name}"
            if feat_name in features:
                img = (LS_seasonal.filterDate(s_date, e_date)
                                  .select(band)
                                  .median()
                                  .unmask(0)
                                  .rename(feat_name)
                                  .clip(roi))
                images_to_stack.append(img)

    # Annual Spectral Indices (SI, RI, TGSI)
    annual_mean    = LS.select(['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']).mean()
    annual_indices = add_derived_indices(annual_mean)
    needed_annual  = [b for b in annual_indices.bandNames().getInfo() if b in features]
    if needed_annual:
        images_to_stack.append(annual_indices.select(needed_annual).clip(roi))

    # Climate
    if 'temp' in features:
        temp = (ee.ImageCollection("MODIS/061/MOD11A2")
                  .select('LST_Day_1km')
                  .filterDate(START_DATE, END_DATE)
                  .filterBounds(roi)
                  .mean()
                  .multiply(0.002)
                  .rename('temp')
                  .clip(roi))
        images_to_stack.append(temp)

    if 'precipitation' in features:
        preci = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                   .filterDate(START_DATE, END_DATE)
                   .filterBounds(roi)
                   .mean()
                   .rename('precipitation')
                   .clip(roi))
        images_to_stack.append(preci)

    # Topography
    elevation = ee.Image("USGS/SRTMGL1_003").select('elevation').clip(roi)
    if 'elevation' in features:
        images_to_stack.append(elevation)
    if 'slope' in features:
        images_to_stack.append(ee.Terrain.slope(elevation).rename('slope'))

    # Soil (SoilGrids)
    for prop in ['sand', 'silt', 'clay', 'phh2o']:
        soil_img = load_soil_image(prop, features, roi)
        if soil_img:
            images_to_stack.append(soil_img)

    final_stack = ee.Image(images_to_stack).toFloat()
    print(f"  ✅ Feature stack ready. Bands: {final_stack.bandNames().getInfo()}")

    # --- Classify & Apply Cropland Mask ---
    classified = final_stack.classify(classifier)

    # lulc      = ee.Image("projects/corestack-datasets/assets/datasets/LULC_v3_river_basin/pan_india_lulc_v3_2023_2024").select('predicted_label')
    lulc = ee.Image(config['gee']['lulc_asset_path']).select('predicted_label')
    # crop_mask = lulc.remap([8, 9, 10, 11], [1, 1, 1, 1], 0)
    # final_soil_map = classified.updateMask(crop_mask).clip(roi)
    
    # Create a dynamic remap list based on the number of crop labels
    target_values = [1] * len(CROP_LABELS) 
    crop_mask = lulc.remap(CROP_LABELS, target_values, 0)
    
    final_soil_map = classified.updateMask(crop_mask).clip(roi)

    # --- Submit Export Task ---
    print(f"  🚀 Submitting export task → {ASSET_ID}")
    try:
        task = ee.batch.Export.image.toAsset(
            image=final_soil_map,
            description=f"Export_{ASSET_NAME}",
            assetId=ASSET_ID,
            region=roi.bounds(),
            scale=30,
            maxPixels=1e13
        )
        task.start()
        print(f"  ✅ Task submitted successfully!\n")
        submitted_count += 1
    except Exception as e:
        print(f"  ❌ Task submission failed: {e}\n")
        failed_count += 1

    # Small delay to avoid GEE rate limits when submitting many tasks
    time.sleep(2)

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------
print(f"\n{'='*60}")
print(f"  BATCH INFERENCE COMPLETE")
print(f"  ✅ Tasks Submitted  : {submitted_count}")
print(f"  ⏭️  Skipped (Exists) : {skipped_count}")
print(f"  ❌ Failed           : {failed_count}")
print(f"  Monitor tasks at   : https://code.earthengine.google.com/tasks")
print(f"{'='*60}")

/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


✅ Google Earth Engine initialized successfully!

Found 72 model file(s) for inference.

  AEZ: 10  |  Nutrient: K  |  Date: 2026-03-11
  Features (16): ['temp', 'SI', 'RI', 'precipitation', 'sand05', 'silt05', 'sand515', 'TGSI', 'NDVI_Kharif', 'NDVI_Rabi', 'NDVI_Zaid', 'pH_0-5', 'pH_5-15', 'NIRv_Kharif', 'NIRv_Rabi', 'NIRv_Zaid']
  ✅ Classifier loaded from: rf_trees_t10_d16_2026-03-11_AEZ_10_K.csv
  ⏳ Building feature stack...
  ✅ Feature stack ready. Bands: ['NIRv_Kharif', 'NIRv_Rabi', 'NIRv_Zaid', 'NDVI_Kharif', 'NDVI_Rabi', 'NDVI_Zaid', 'SI', 'RI', 'TGSI', 'temp', 'precipitation', 'sand05', 'sand515', 'silt05', 'pH_0_5', 'pH_5_15']
  🚀 Submitting export task → projects/ee-mtpictd-ratinder-mcs/assets/AEZ_10_K_20260311
  ✅ Task submitted successfully!

  AEZ: 10  |  Nutrient: N  |  Date: 2026-03-11
  Features (16): ['temp', 'SI', 'RI', 'precipitation', 'sand05', 'silt05', 'sand515', 'TGSI', 'NDVI_Kharif', 'NDVI_Rabi', 'NDVI_Zaid', 'pH_0-5', 'pH_5-15', 'NIRv_Kharif', 'NIRv_Rabi', 'NIRv